In [ ]:
import sys
import re
from datetime import date

print('=' * 60, file=sys.stderr)
print('Water Treatment OpCo Financial Model', file=sys.stderr)
print('=' * 60, file=sys.stderr)

# ================================================================
# 1. TIMELINE AND INDEXATION FACTORS
# ================================================================
# Quarterly model: 60 quarters, Q1-2016 through Q4-2030.
# Quarter end dates: Mar 31, Jun 30, Sep 30, Dec 31 each year.

quarter_end_dates = []
for year in range(2016, 2031):
    for m, d in [(3, 31), (6, 30), (9, 30), (12, 31)]:
        quarter_end_dates.append(date(year, m, d))

N_Q = len(quarter_end_dates)  # 60

# IF1: 1.00 at 1 Jan 2016, +2.2% every subsequent 1 January.
# IF2: 1.00 at 1 Jan 2016, +4.0% every subsequent 1 January.
# IF4: 1.00 at 1 Jan 2016, +2.5% every subsequent 1 January.
#   -> For all quarter-end dates in year Y, factor = (1+r)^(Y-2016).
#
# IF3: 1.00 at 1 Jan 2016, +4.8% every 1 July.
# IF5: 1.00 at 1 Jan 2016, +2.0% every 1 July.
#   -> Count how many Jul 1 dates (2016..2030) are <= quarter-end date.

IF1 = [(1.022 ** (qd.year - 2016)) for qd in quarter_end_dates]
IF2 = [(1.04  ** (qd.year - 2016)) for qd in quarter_end_dates]
IF4 = [(1.025 ** (qd.year - 2016)) for qd in quarter_end_dates]

def count_jul1s(qd):
    n = 0
    for y in range(2016, 2031):
        if date(y, 7, 1) <= qd:
            n += 1
    return n

IF3 = [(1.048 ** count_jul1s(qd)) for qd in quarter_end_dates]
IF5 = [(1.02  ** count_jul1s(qd)) for qd in quarter_end_dates]

print(f'Timeline: {N_Q} quarters, {quarter_end_dates[0]} to {quarter_end_dates[-1]}', file=sys.stderr)
print(f'IF1[0]={IF1[0]:.4f}, IF3[0]={IF3[0]:.4f} (should be 1.0)', file=sys.stderr)
print(f'IF3[2]={IF3[2]:.4f} (Q3-2016, should be 1.048)', file=sys.stderr)

In [ ]:
# ================================================================
# 2. EXPENSES, LIFECYCLE COSTS, REVENUE COEFFICIENTS
# ================================================================

# Per-quarter expenses
labour    = [2_500_000.0 * IF3[i] for i in range(N_Q)]
materials = [1_750_000.0 * IF4[i] for i in range(N_Q)]
other_exp = [  750_000.0 * IF5[i] for i in range(N_Q)]
profit_margin = [0.09 * (labour[i] + materials[i] + other_exp[i]) for i in range(N_Q)]
total_opex = [labour[i] + materials[i] + other_exp[i] + profit_margin[i] for i in range(N_Q)]

# Lifecycle costs (not escalated, no profit margin)
lifecycle = [0.0] * N_Q
for i, qd in enumerate(quarter_end_dates):
    if qd == date(2021, 12, 31):
        lifecycle[i] = 10_000_000.0
    elif qd == date(2028, 6, 30):
        lifecycle[i] = 5_000_000.0

# Total costs = opex + lifecycle
total_costs = [total_opex[i] + lifecycle[i] for i in range(N_Q)]

# Revenue coefficients: Revenue_q = BP1 * rc1[q] + BP2 * rc2[q]
rc1 = [0.25 * IF1[i] for i in range(N_Q)]
rc2 = [0.25 * IF2[i] for i in range(N_Q)]

# ---------- Q1: Total Labour in 2019 ----------
labour_2019 = sum(labour[i] for i, qd in enumerate(quarter_end_dates) if qd.year == 2019)
print(f'Q1: Total Labour 2019 = ${labour_2019:,.2f}', file=sys.stderr)

# ---------- Q2: Profit Margin in Q1-2025 (ending 31 Mar 2025) ----------
idx_q2 = quarter_end_dates.index(date(2025, 3, 31))
pm_q1_2025 = profit_margin[idx_q2]
print(f'Q2: Profit Margin Q1-2025 = ${pm_q1_2025:,.2f}', file=sys.stderr)

# ---------- Q3: Total of all Expenses + Lifecycle Costs ----------
total_all = sum(total_costs)
print(f'Q3: Total all costs = ${total_all:,.2f}', file=sys.stderr)

In [ ]:
# ================================================================
# 3. CASH FLOW SIMULATION ENGINE
# ================================================================

def simulate(bp1, bp2, debt_rate=0.03):
    """
    Run quarterly cash flow simulation.
    Each quarter:
      - Interest accrues on opening debt at debt_rate per quarter.
      - Revenue received, expenses + lifecycle + interest paid.
      - Shortfall borrowed; excess repays debt.
    Returns (final_balance, total_interest_paid).
    final_balance = cash - 5M_return - remaining_debt.
    """
    cash = 5_000_000.0
    debt = 0.0
    total_interest = 0.0
    for i in range(N_Q):
        interest = debt * debt_rate
        total_interest += interest
        revenue = bp1 * rc1[i] + bp2 * rc2[i]
        cash += revenue - total_costs[i] - interest
        if cash < 0:
            debt -= cash
            cash = 0.0
        if cash > 0 and debt > 0:
            repay = min(cash, debt)
            cash -= repay
            debt -= repay
    return cash - 5_000_000.0 - debt, total_interest


def solve_bp2(bp1, debt_rate=0.03):
    """Binary search for BP2 giving zero final balance."""
    lo, hi = 0.0, 100_000_000.0
    for _ in range(200):
        mid = (lo + hi) / 2.0
        bal, _ = simulate(bp1, mid, debt_rate)
        if bal < 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0


def compute_ads(bp1, bp2):
    """ADS = sum_i |Revenue_i - Expenses_i - Lifecycle_i|."""
    return sum(abs(bp1 * rc1[i] + bp2 * rc2[i] - total_costs[i]) for i in range(N_Q))


# ---------- Q4: BP1=$9M, last 5 digits of BP2 ----------
bp2_q4 = solve_bp2(9_000_000.0)
last5_q4 = round(bp2_q4) % 100_000
print(f'Q4: BP2={bp2_q4:,.2f}, last5={last5_q4}', file=sys.stderr)

# ---------- Q5: BP2=2.5*BP1, combined to nearest $10K ----------
lo5, hi5 = 0.0, 50_000_000.0
for _ in range(200):
    m5 = (lo5 + hi5) / 2.0
    b5, _ = simulate(m5, 2.5 * m5)
    if b5 < 0: lo5 = m5
    else: hi5 = m5
bp1_q5 = (lo5 + hi5) / 2.0
combined_q5 = 3.5 * bp1_q5
combined_rounded = round(combined_q5 / 10_000) * 10_000
print(f'Q5: combined={combined_q5:,.2f}, rounded={combined_rounded:,}', file=sys.stderr)

In [ ]:
# ================================================================
# 4. QUESTIONS 6-9: ADS MINIMIZATION (BP1 multiple of $500K)
# ================================================================

best_ads_q6 = float('inf')
best_bp1_q6 = None
best_bp2_q6 = None

for bp1_test in range(500_000, 20_000_001, 500_000):
    bp2_test = solve_bp2(float(bp1_test))
    if bp2_test < -0.01:
        continue
    bp2_test = max(bp2_test, 0.0)
    ads_val = compute_ads(float(bp1_test), bp2_test)
    if ads_val < best_ads_q6:
        best_ads_q6 = ads_val
        best_bp1_q6 = float(bp1_test)
        best_bp2_q6 = bp2_test

print(f'Q6: best BP1 = ${best_bp1_q6:,.0f}', file=sys.stderr)

ads_rounded_q7 = round(best_ads_q6 / 1000) * 1000
print(f'Q7: ADS = ${best_ads_q6:,.2f}, rounded = ${ads_rounded_q7:,}', file=sys.stderr)

_, total_int_q8 = simulate(best_bp1_q6, best_bp2_q6)
print(f'Q8: Total interest = ${total_int_q8:,.2f}', file=sys.stderr)

ratio_pct_q9 = (best_bp2_q6 / best_bp1_q6) * 100.0
ratio_dec_q9 = int(round(ratio_pct_q9 * 100)) % 100
print(f'Q9: BP2/BP1 = {ratio_pct_q9:.2f}%, dec digits = {ratio_dec_q9:02d}', file=sys.stderr)

In [ ]:
# ================================================================
# 5. QUESTIONS 10-11: ZERO INTEREST, UNCONSTRAINED L1 MINIMIZATION
# ================================================================
# With zero interest: sum(revenues) = sum(costs) => BP2 = (TC - BP1*S1)/S2.
# ADS is piecewise linear in BP1. Its minimum is at a breakpoint where
# one |affine_i| term switches sign.
#
# D_i(BP1) = BP1 * a_i + c_i  where:
#   a_i = rc1_i - S1*rc2_i/S2
#   c_i = TC*rc2_i/S2 - costs_i

S1 = sum(rc1)
S2 = sum(rc2)
TC = sum(total_costs)

a_coeff = [rc1[i] - S1 * rc2[i] / S2 for i in range(N_Q)]
c_coeff = [TC * rc2[i] / S2 - total_costs[i] for i in range(N_Q)]

bp1_max_10 = TC / S1  # upper bound where BP2 = 0

# Collect all breakpoints in feasible range
breakpoints = {0.0, bp1_max_10}
for i in range(N_Q):
    if abs(a_coeff[i]) > 1e-15:
        bp = -c_coeff[i] / a_coeff[i]
        if 0 <= bp <= bp1_max_10:
            breakpoints.add(bp)

breakpoints = sorted(breakpoints)
print(f'Breakpoints: {len(breakpoints)}', file=sys.stderr)

# Evaluate ADS at each breakpoint
best_bp1_10, best_ads_10 = 0.0, float('inf')
for bp in breakpoints:
    a = sum(abs(bp * a_coeff[i] + c_coeff[i]) for i in range(N_Q))
    if a < best_ads_10:
        best_ads_10 = a
        best_bp1_10 = bp

bp2_q10 = (TC - best_bp1_10 * S1) / S2
ads_last3_q10 = round(best_ads_10) % 1000
print(f'Q10: BP1={best_bp1_10:,.4f}, BP2={bp2_q10:,.4f}', file=sys.stderr)
print(f'Q10: ADS={best_ads_10:,.2f}, last3={ads_last3_q10}', file=sys.stderr)

bp1_rounded_q11 = round(best_bp1_10)
digit_sum_q11 = sum(int(d) for d in str(bp1_rounded_q11))
print(f'Q11: BP1 rounded={bp1_rounded_q11}, digit sum={digit_sum_q11}', file=sys.stderr)

In [ ]:
# ================================================================
# 6. MATCH TO MULTIPLE CHOICE AND BUILD ANSWERS DICT
# ================================================================

data_dir = '/workspace/data'


def match_mc(question_file, value):
    """Return the MC letter whose numeric option is closest to value."""
    with open(question_file) as f:
        text = f.read()
    options = {}
    for m in re.finditer(r'([a-fA-F])\.\s+(.+)', text):
        letter = m.group(1).upper()
        cleaned = m.group(2).strip().replace('$', '').replace(',', '').replace('%', '')
        try:
            options[letter] = float(cleaned)
        except ValueError:
            pass
    best, best_dist = None, float('inf')
    for letter, opt_val in options.items():
        dist = abs(value - opt_val)
        if dist < best_dist:
            best_dist = dist
            best = letter
    return best


q1_ans  = match_mc(f'{data_dir}/question1.txt',  labour_2019)
q2_ans  = match_mc(f'{data_dir}/question2.txt',  pm_q1_2025)
q3_ans  = match_mc(f'{data_dir}/question3.txt',  total_all)
q4_ans  = match_mc(f'{data_dir}/question4.txt',  last5_q4)
q5_ans  = match_mc(f'{data_dir}/question5.txt',  combined_rounded)
q6_ans  = match_mc(f'{data_dir}/question6.txt',  best_bp1_q6)
q7_ans  = match_mc(f'{data_dir}/question7.txt',  ads_rounded_q7)
q8_ans  = match_mc(f'{data_dir}/question8.txt',  total_int_q8)
q9_ans  = match_mc(f'{data_dir}/question9.txt',  ratio_dec_q9)
q10_ans = match_mc(f'{data_dir}/question10.txt', ads_last3_q10)
q11_ans = match_mc(f'{data_dir}/question11.txt', digit_sum_q11)

answers = {
    'question1':  q1_ans,
    'question2':  q2_ans,
    'question3':  q3_ans,
    'question4':  q4_ans,
    'question5':  q5_ans,
    'question6':  q6_ans,
    'question7':  q7_ans,
    'question8':  q8_ans,
    'question9':  q9_ans,
    'question10': q10_ans,
    'question11': q11_ans,
}

print('\nFinal answers:', file=sys.stderr)
for k in sorted(answers, key=lambda x: int(x.replace('question', ''))):
    print(f'  {k}: {answers[k]}', file=sys.stderr)